# BayFlux EMU Demonstration Notebook
By Tyler W. H. Backman

This notebook reproduces the first example in Antoniewicz 2006 using the bayflux EMU simulator built on COBRApy and NumPy. 

Inputs, all specified as Python code within this notebook:
* COBRApy model which includes 6 metabolites and 5 reactions
* atom transitions for all 5 reactions listed above
* a flux vector to simulate
* input metabolite labeling (metabolite A)
* an output metabolite to simulate (metabolite F)

Outputs:
* a compiled set of operations to rapidly simulate labeling for any flux vector
* simulated labeling for metabolite F

Notes:
* All functions use numpy matrices or Python lists and tuples, but we display them here as Pandas DataFrames in order to label the rows and columns for illustration purposes.
* EMUs are specified in code with zero-indexed atoms, but are printed as text in output using repr() as 1-indexed, for direct comparison to published literature.

In [ ]:
# Note: for use on top of the tbackman/debian-cheminformatics image on dockerhub
# install the development version of jQMM2
# as follows: pip3 install -e ../

In [ ]:
import cobra
import bayflux
import pandas as pd
import numpy as np

### First we will use COBRApy to build a model corresponding to the reaction stoichiometry in Fig. 3 of Antoniewicz 2006 which consists of 6 metabolites named A-F, and 5 reactions

<img src="input_data/fig3.png" alt="" style="width: 300px;"/>

In [ ]:
# create a blank model
model = cobra.Model('fig3')

# define metabolites
A = cobra.Metabolite(
    'A',
    formula='C3',
    name='A',
    compartment='c')
B = cobra.Metabolite(
    'B',
    formula='C3',
    name='B',
    compartment='c')
C = cobra.Metabolite(
    'C',
    formula='C2',
    name='C',
    compartment='c')
D = cobra.Metabolite(
    'D',
    formula='C3',
    name='D',
    compartment='c')
E = cobra.Metabolite(
    'E',
    formula='C1',
    name='E',
    compartment='c')
F = cobra.Metabolite(
    'F',
    formula='C3',
    name='F',
    compartment='c')

# define reactions and add them to the model
a_b = cobra.Reaction('a_b')
a_b.lower_bound = 0
a_b.upper_bound = 500
a_b.add_metabolites({
    A: -1.0,
    B: 1.0
})
print(a_b.reaction)

b_ec = cobra.Reaction('b_ec')
b_ec.lower_bound = 0
b_ec.upper_bound = 500
b_ec.add_metabolites({
    B: -1.0,
    E: 1.0,
    C: 1.0
})
print(b_ec.reaction)

bc_de = cobra.Reaction('bc_de')
bc_de.lower_bound = 0
bc_de.upper_bound = 500
bc_de.add_metabolites({
    B: -1.0,
    C: -1.0,
    D: 1.0,
    E: 2.0
})
print(bc_de.reaction)

d_f = cobra.Reaction('d_f')
d_f.lower_bound = 0
d_f.upper_bound = 500
d_f.add_metabolites({
    D: -1.0,
    F: 1.0
})
print(d_f.reaction)

b_d = cobra.Reaction('b_d')
b_d.lower_bound = -500
b_d.upper_bound = 500
b_d.add_metabolites({
    B: -1.0,
    D: 1.0
})
print(b_d.reaction)

model.add_reactions([a_b, b_ec, bc_de, d_f, b_d])

Now we convert this model to a bayflux ReactionNetwork, which inherets 'EnhancedReaction' objects from each cobra.Reaction, allowing us to add in atom transitions

In [ ]:
model = bayflux.ReactionNetwork(model)
model

Here we set the atom transitions for each reaction, and show an example of viewing an EnhancedReaction which reports atom transitions

In [ ]:
# create dict of metabolites by name
# we use this instead of directly using the metabolite IDs above, because the conversion to
# a bayflux.ReactionNetwork created new metabolite objects
m = {m.id:m for m in model.metabolites}

model.reactions.a_b.transitions = [bayflux.AtomTransition(
        ((m['A'], [1,2,3]),), # reactant labels
        ((m['B'], [1,2,3]),) # product labels
    )]
model.reactions.b_ec.transitions = [bayflux.AtomTransition(
        ((m['B'], [1,2,3]),), # reactant labels
        ((m['E'], [1]), (m['C'], [2,3]),) # product labels
     )]
model.reactions.bc_de.transitions = [bayflux.AtomTransition(
        ((m['B'], [1,2,3]), (m['C'], [4,5]),), # reactant labels
        ((m['E'], [1]), (m['D'], [2,3,4]), (m['E'], [5]),) # product labels
     )]
model.reactions.d_f.transitions = [bayflux.AtomTransition(
        ((m['D'], [1,2,3]),), # reactant labels
        ((m['F'], [1,2,3]),) # product labels
     )]
model.reactions.b_d.transitions = [bayflux.AtomTransition(
        ((m['B'], [1,2,3]),), # reactant labels
        ((m['D'], [1,2,3]),) # product labels
     )]

model.reactions.b_d

jQMM 1.0 like transition printing

In [ ]:
model.reactions.b_d.transitions[0]

Here we represent fluxes as a FluxVector object, with fluxes ordered to match the model variables

**Note**: non-reversible reactions should always carry zero reverse flux!

In [ ]:
fluxDict = {'a_b': [100, 0], 'b_d': [110, 50], 'b_ec': [20, 0], 'bc_de': [20, 0], 'd_f': [80, 0]}
fluxes = bayflux.FluxVector(model, fluxDict)
fluxes

### Perform a test to find all EMU reactions producing metabolite F

In [ ]:
F = model.metabolites.get_by_id('F')
outputEMU = bayflux.EMU(F,[0,1,2])
result = bayflux.findProducingEMUTransitions(outputEMU)
result

### perform test full EMU decomposition of metabolite F precursors

In [ ]:
fullDecomposition = bayflux.emuDecomposition([outputEMU])
fullDecomposition

### Split apart EMU reactions by product size

In [ ]:
transitionsBySize = bayflux.splitBySize(fullDecomposition)
transitionsBySize

###  EMU simulation matrix descriptions

AX=BY
X=A^-1BY

Matrix A: fluxes (internal only, not inputs)
* each column represents fluxes into and out of a given metabolite
* order from top to bottom is fluxes from EMU reactions towards a given other metabolite,
in the same order as X top to bottom
* a metabolite's own cell represents fluxes towards that metabolite from all reactions
* values are fluxes away from the metabolite in that column (e.g. fluxes towards it are negative)

Matrix B (input fluxes only):
* columns are external EMUs from feed
* rows are internal EMUs
* values represent feed fluxes from feed reaction

Matrix Y: input labeling
* each row represents an external EMU with the same order as the columns in Matrix B
* columns represent the number of additional neutrons (0, 1, 2, etc.)
* values represent the mass distribution of each external EMU distributed over the columns

Matrix X: predicted labeling
* each row represents a predicted EMU with the same order as Matrix A
* columns represent the number of additional neutrons (0, 1, 2, etc.)
* values represent the mass distribution fir each predicted EMU

### Test identifying the internal and external EMUs for all size 1 EMU reactions, which is later used as the coordinates for the resulting matrices

In [ ]:
matrixCoords = bayflux.constructMatrixCoords(transitionsBySize[1])
matrixCoords

### Create example zero matrices for the length-1 EMU reactions

In [ ]:
exampleMatrices = bayflux.constructZeroMatrices(matrixCoords)
print('Matrix A')
display(pd.DataFrame(exampleMatrices[0], index=matrixCoords['internalEMUs']['text'], columns=matrixCoords['internalEMUs']['text']))
print('Matrix B')
display(pd.DataFrame(exampleMatrices[1], index=matrixCoords['internalEMUs']['text'], columns=matrixCoords['externalEMUs']['text']))

### Example of compiling the size 1 EMU reactions into a set of matrix addition operations

In [ ]:
operations = bayflux.compileEMUTransitionList(transitionsBySize[1], matrixCoords, model)
pd.DataFrame(operations, columns=('A=0/B=1 matrix selector', 'Row', 'Column', 'Flux selector', 'Factor'))

### Now we will compile and execute all A and B matricies for each EMU sub-network size

In this example we make matY semi-manually

#### Size 1 sub-network

In [ ]:
# compute A and B matrix for size 1
matrixCoords1 = bayflux.constructMatrixCoords(transitionsBySize[1])
operations1 = bayflux.compileEMUTransitionList(transitionsBySize[1], matrixCoords1, model)
zeroMatrices1 = bayflux.constructZeroMatrices(matrixCoords1)
result1 = bayflux.executeCompiledEMUTransitionList(operations1, zeroMatrices1, fluxes)

print('Matrix A')
display(pd.DataFrame(result1[0], index=matrixCoords1['internalEMUs']['text'], columns=matrixCoords1['internalEMUs']['text']))
print('Matrix B')
display(pd.DataFrame(result1[1], index=matrixCoords1['internalEMUs']['text'], columns=matrixCoords1['externalEMUs']['text']))

In [ ]:
# compute input matY for size 1
matY1 = np.matrix([[0, 1], [1, 0]])
pd.DataFrame(matY1, index=matrixCoords1['externalEMUs']['text'])

In [ ]:
# compute predicted labeling matX for size 1
matA = result1[0]
matB = result1[1]
matX1 = np.array(np.linalg.inv(matA).dot(matB).dot(matY1))
pd.DataFrame(matX1, index=matrixCoords1['internalEMUs']['text'])

#### Size 2 sub-network

In [ ]:
# compute A and B matrix for size 2
matrixCoords2 = bayflux.constructMatrixCoords(transitionsBySize[2])
operations2 = bayflux.compileEMUTransitionList(transitionsBySize[2], matrixCoords2, model)
zeroMatricies2 = bayflux.constructZeroMatrices(matrixCoords2)
result2 = bayflux.executeCompiledEMUTransitionList(operations2, zeroMatricies2, fluxes)

print('Matrix A')
display(pd.DataFrame(result2[0], index=matrixCoords2['internalEMUs']['text'], columns=matrixCoords2['internalEMUs']['text']))
print('Matrix B')
display(pd.DataFrame(result2[1], index=matrixCoords2['internalEMUs']['text'], columns=matrixCoords2['externalEMUs']['text']))

In [ ]:
# compute input matY for size 2
# matY2 = pd.DataFrame(0, columns=range(0,3), index=matrixCoords2['externalEMUs']['text'])
matY2 = np.zeros(shape=(len(matrixCoords2['externalEMUs']['text']), 3))
matY2[0] = (0, 1, 0)
matY2[1] = np.convolve(
    matX1[matrixCoords1['internalEMUs']['text'].index('B[3]')].tolist()[0],
    matX1[matrixCoords1['internalEMUs']['text'].index('C[1]')].tolist()[0]
)

pd.DataFrame(matY2, index=matrixCoords2['externalEMUs']['text'])

In [ ]:
# compute predicted labeling matX for size 2
matA = result2[0]
matB = result2[1]
matX2 = np.array(np.linalg.inv(matA).dot(matB).dot(matY2))

pd.DataFrame(matX2, index=matrixCoords2['internalEMUs']['text'])

#### Size 3 sub-network

In [ ]:
# compute A and B matrix for size 3
matrixCoords3 = bayflux.constructMatrixCoords(transitionsBySize[3])
operations3 = bayflux.compileEMUTransitionList(transitionsBySize[3], matrixCoords3, model)
zeroMatricies3 = bayflux.constructZeroMatrices(matrixCoords3)
result3 = bayflux.executeCompiledEMUTransitionList(operations3, zeroMatricies3, fluxes)

print('Matrix A')
display(pd.DataFrame(result3[0], index=matrixCoords3['internalEMUs']['text'], columns=matrixCoords3['internalEMUs']['text']))
print('Matrix B')
display(pd.DataFrame(result3[1], index=matrixCoords3['internalEMUs']['text'], columns=matrixCoords3['externalEMUs']['text']))

In [ ]:
# compute input matY for size 3
matY3 = np.zeros(shape=(len(matrixCoords3['externalEMUs']['text']), 4))
matY3[0] = (0, 1, 0, 0)
# matY3[1] = np.convolve(matX2.loc['B[2, 3]'], matX1.loc['C[1]'])
matY3[1] = np.convolve(
    matX2[matrixCoords2['internalEMUs']['text'].index('B[2, 3]')].tolist()[0],
    matX1[matrixCoords1['internalEMUs']['text'].index('C[1]')].tolist()[0]
)

pd.DataFrame(matY3, index=matrixCoords3['externalEMUs']['text'])

In [ ]:
# compute predicted labeling matX for size 3
matA = result3[0]
matB = result3[1]
matX3 = np.array(np.linalg.inv(matA).dot(matB).dot(matY3))

pd.DataFrame(matX3, index=matrixCoords3['internalEMUs']['text'])

### Now we define the substrate labeling vector, and test the ability to extract a mass distribution vector from this labeling

In [ ]:
# demo creation of a mass distribution vector for a binary labeling vector

substrateLabelingDict = {
    m['A']: ((1.0, [0, 1, 0]),)
}

inputEMU = bayflux.EMU(m['A'], [0,1,2])
extractedLabeling = bayflux.extractSubstrateEMU(inputEMU, substrateLabelingDict)
extractedLabeling

### Run fully automated EMU simulation

#### First let's decompose the EMU network, and compile its structure into a list of matrix addition operations

In [ ]:
# EMUs to simulate in list form
emusToSimulate = [bayflux.EMU(m['F'],[0,1,2])]

compiledData = bayflux.emuCompile(emusToSimulate, model, substrateLabelingDict, True, True)
compiledData

#### test automated creation of a matrix Y

In [ ]:
results = {1: matX1, 2: matX2, 3: matX3}
substrateMDVs, operationsY = bayflux.compileMatrixY(compiledData, compiledData[2]['matrixCoords'], 3, substrateLabelingDict)
autoMatY3 = bayflux.executeCompiledMatrixY(substrateMDVs, operationsY, results, 3)
pd.DataFrame(autoMatY3, index=compiledData[2]['matrixCoords']['externalEMUs']['text'])

#### Now we'll test the injection of a specific flux vector, in order to simulate the labeling of metabolite F

In [ ]:
fluxes

In [ ]:
results = bayflux.simulateLabeling(compiledData, fluxes, substrateLabelingDict)
results

#### Now view final results and compare to published metabolite F MDV of [0.0001, 0.8008, 0.1983, 0.0009]

In [ ]:
pd.DataFrame(results[3], index=compiledData[2]['matrixCoords']['internalEMUs']['text'])